# Final Project: Training with fewer comments
Select **A100 GPU** in Runtime → Change runtime type, then run these cells in order. The longer code is in GitHub’s FinalProject folder.

This notebook pins the completed training implementation. The latest live, generator, study and chunking tools are in the [platform README](https://github.com/S3eeDTR/LLM-AuthorBench-Assignment/tree/main/FinalProject). Reuse downloaded checkpoints to run the platform without retraining.

## 1. Get the project from GitHub
Download the fixed code version used for this experiment. Rerunning this cell keeps the same version.

In [ ]:
from pathlib import Path
import subprocess
REPO = Path('/content/authorbench_final_project')
if not REPO.exists():
    subprocess.run(['git', 'clone', '-q', 'https://github.com/S3eeDTR/LLM-AuthorBench-Assignment.git', str(REPO)], check=True)
subprocess.run(['git', '-C', str(REPO), 'checkout', '-q', '--detach', '2e9104d86defa2e67c32154ff9687bfde8821d6a'], check=True)
CODE = REPO / 'FinalProject'
print('Project downloaded:', CODE)


## 2. Install packages
Keep Colab’s GPU PyTorch. If Colab requests a restart, restart the session and rerun cells 1–3.

In [ ]:
%pip -q install -r /content/authorbench_final_project/FinalProject/requirements-colab.txt


## 3. Check the GPU
This run needs an A100. Both experiments use 10 epochs, 512 tokens and an effective batch of 32.

In [ ]:
import os, sys
os.environ['USE_TF'] = '0'
sys.path.insert(0, str(CODE))
import experiment as exp
from google.colab import files
from IPython.display import Image, display
exp.WORK = Path('/content/FinalProject')
exp.require_gpu()
print('GPU:', exp.torch.cuda.get_device_name(0))
print('Epochs:', exp.EPOCHS, '| Tokens:', exp.MAX_LENGTH, '| Effective batch:', exp.MICRO_BATCH * exp.ACCUMULATION)


## 4. Load the data and completed baselines
Use the Assignment 1 split and the saved SVM/CodeT5 results. File hashes and test rows are checked automatically.

In [ ]:
exp.download_data()
frames = exp.load_data()
baselines = exp.load_baselines(CODE / 'baseline_reference.zip', frames)
print({part: len(frame) for part, frame in frames.items()})
display(baselines[['model', 'variant', 'accuracy', 'macro_f1']])


## 5. Prepare model inputs
Tokenize original and stripped code once. This downloads the pinned CodeT5 tokenizer.

In [ ]:
tokenizer, tokens = exp.prepare_tokens(frames)


## 6. Train: 50% comment removal
Strip half the training programs each epoch. Watch the progress bar and validation scores; this starts a fresh pretrained model.

In [ ]:
mixed_folder = exp.run_training(0.5, frames, tokens, tokenizer)


## 7. Train: 100% comment removal
Strip all training programs for the ablation. This also starts a fresh pretrained model.

In [ ]:
stripped_folder = exp.run_training(1.0, frames, tokens, tokenizer)


## 8. Evaluate both models
Run only after both training cells finish. Evaluate the best validation checkpoints on original and stripped test code.

In [ ]:
results = exp.evaluate_both(frames, tokens, tokenizer)
print(exp.json.dumps(results, indent=2))


## 9. Compare results
Compare all four models. Report stripped-code Macro-F1 gains alongside changes in original-code accuracy.

In [ ]:
comparison, tradeoffs = exp.comparison_table(baselines)
display(comparison)
display(tradeoffs)
exp.save_figures(comparison)
display(Image(filename=str(exp.WORK / 'results/accuracy_comparison.png')))
display(Image(filename=str(exp.WORK / 'results/macro_f1_comparison.png')))


## 10. Download the evidence
Save the ZIP locally. It contains settings, training logs, predictions, metrics and charts, but excludes large model weights.

In [ ]:
files.download(str(exp.evidence_bundle()))


## 11. Optional: save model weights
Uncomment either line to download a checkpoint (about 1.34 GB each). The evidence ZIP alone cannot restore a model.

In [ ]:
# files.download(str(mixed_folder / "best_model.pt"))
# files.download(str(stripped_folder / "best_model.pt"))


## Keep in mind
Allow roughly 2–3 hours on A100 for both runs; timing varies. Colab storage is temporary and interrupted training cannot resume.

One seed and a previously examined test set limit the conclusions. Keep the settings fixed; CPU SVM and GPU training times are not directly comparable.